In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BASE_PATH = Path('.')
FEATURES_DIR = BASE_PATH / 'features'
MODEL_PATH = BASE_PATH / 'model_behavioral.pth'

In [ ]:
class BehavioralDataset(Dataset):
    def __init__(self, sequences, labels, augment=False, noise_std=0.05):
        self.sequences = sequences.astype(np.float32)
        self.labels = labels.astype(np.int64)
        self.augment = augment
        self.noise_std = noise_std

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx].copy()
        
        if self.augment:
            noise = np.random.normal(0, self.noise_std, seq.shape)
            seq = seq + noise
        
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

print('Dataset class defined')

In [ ]:
class BehavioralLSTM(nn.Module):
    def __init__(self, input_size=10, hidden_size=32, num_layers=1, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=0.0 if num_layers == 1 else dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        x = self.dropout(last_out)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

print('BehavioralLSTM model defined')

In [ ]:
print('Loading preprocessed sequences...')
print()

all_sequences = []
all_labels = []

virat_dir = FEATURES_DIR / 'virat'
ucf_dir = FEATURES_DIR / 'ucf'

print('Loading VIRAT sequences (Label=0: Normal)...')
virat_count = 0
if virat_dir.exists():
    for npy_file in sorted(virat_dir.glob('*.npy')):
        seq = np.load(npy_file)
        all_sequences.append(seq)
        all_labels.append(0)
        virat_count += 1

print(f'  ✓ Loaded {virat_count} VIRAT sequences')

print('Loading UCF sequences (Label=1: Suspicious - Shoplifting/Robbery)...')
ucf_count = 0
if ucf_dir.exists():
    for npy_file in sorted(ucf_dir.glob('*.npy')):
        seq = np.load(npy_file)
        all_sequences.append(seq)
        all_labels.append(1)
        ucf_count += 1

print(f'  ✓ Loaded {ucf_count} UCF sequences')

X_all = np.array(all_sequences, dtype=np.float32)
y_all = np.array(all_labels, dtype=np.int64)

print()
print(f'Dataset shape: {X_all.shape}')
print(f'  Sequences: {len(X_all)}')
print(f'  Frames per sequence: {X_all.shape[1] if len(X_all.shape) > 1 else 0}')
print(f'  Features per frame: {X_all.shape[2] if len(X_all.shape) > 2 else 0}')
print()
print(f'Label distribution:')
print(f'  Normal (VIRAT): {(y_all == 0).sum()}')
print(f'  Suspicious (UCF): {(y_all == 1).sum()}')

In [ ]:
shuffle_idx = np.random.RandomState(42).permutation(len(X_all))
X_all = X_all[shuffle_idx]
y_all = y_all[shuffle_idx]

train_size = int(0.7 * len(X_all))
val_size = int(0.15 * len(X_all))

X_train = X_all[:train_size]
y_train = y_all[:train_size]

X_val = X_all[train_size:train_size+val_size]
y_val = y_all[train_size:train_size+val_size]

X_test = X_all[train_size+val_size:]
y_test = y_all[train_size+val_size:]

train_dataset = BehavioralDataset(X_train, y_train, augment=True, noise_std=0.05)
val_dataset = BehavioralDataset(X_val, y_val, augment=False)
test_dataset = BehavioralDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print('Train/Val/Test split (70/15/15):')
print(f'  Train: {len(X_train)} ({(y_train==0).sum()} Normal, {(y_train==1).sum()} Suspicious)')
print(f'  Val:   {len(X_val)} ({(y_val==0).sum()} Normal, {(y_val==1).sum()} Suspicious)')
print(f'  Test:  {len(X_test)} ({(y_test==0).sum()} Normal, {(y_test==1).sum()} Suspicious)')

In [ ]:
model = BehavioralLSTM(input_size=10, hidden_size=32, num_layers=1, dropout=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-4)

print('Model configured:')
print(f'  Architecture: BehavioralLSTM')
print(f'  Input features: 10')
print(f'  Sequence frames: 18')
print(f'  Hidden size: 32')
print(f'  Dropout: 0.5')
print(f'  Optimizer: Adam')
print(f'  Learning rate: 0.0003')
print(f'  Weight decay: 1e-4')
print(f'  Loss: CrossEntropyLoss')

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for sequences, labels in train_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for sequences, labels in val_loader:
            sequences = sequences.to(device)
            labels = labels.to(device)
            
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels

print('Training functions defined')

In [ ]:
print('='*70)
print('TRAINING BEHAVIORAL RISK DETECTION MODEL')
print('='*70)
print()

num_epochs = 40
train_losses = []
train_accs = []
val_losses = []
val_accs = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{num_epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

print()
print('Training completed!')
print(f'Best model saved to: {MODEL_PATH}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_accs, label='Train Accuracy', linewidth=2)
axes[1].plot(val_accs, label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(BASE_PATH / 'training_metrics.png', dpi=150)
print('Training metrics saved to: training_metrics.png')
plt.show()

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0
all_test_preds = []
all_test_labels = []

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        labels = labels.to(device)
        
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_labels.extend(labels.cpu().numpy())

test_loss = test_loss / len(test_loader)
test_acc = test_correct / test_total

print('='*70)
print('TEST RESULTS')
print('='*70)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print()
print('Classification Report:')
print(classification_report(all_test_labels, all_test_preds, target_names=['Normal', 'Suspicious']))
print()
print('Confusion Matrix:')
cm = confusion_matrix(all_test_labels, all_test_preds)
print(cm)